<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zeshengliu213/ai-for-materials/blob/main/01%20-%20Basics/01c_plotting.ipynb)

# 01c · 科研数据可视化：Matplotlib 与 Seaborn

在上一节（`01b`）里，我们用 `describe()` 和 `groupby()` 算出了溶解度随芳香环数变化的均值和标准差。但统计数字有时会掩盖数据的真实长相——比如数据是不是双峰分布？有没有离群的极端分子？两个特征之间是不是非线性的？

无论是建模前**摸清数据规律**，还是建模后**展示预测结果**，都离不开画图。Python 里最核心的两个绘图工具是：
1. **`matplotlib`**：最底层、自由度最高的基础绘图库，小到一根刻度线、大到多子图排版都能精细控制；
2. **`seaborn`**：建立在 `matplotlib` 之上的统计绘图库，直接接收 Pandas 的 `DataFrame`，一两行代码就能画出漂亮的分组分布图、箱线图和相关性热图。

> **小提示**：由于 Google Colab 和部分 Linux 环境默认没有安装中文字体，图上的中文字往往会显示成小方框 `□□□`。因此，我们按照英文期刊的习惯，**图表内的坐标轴标签和标题统一用英文**，正文讲解依然用中文。

### 本节内容
1. Matplotlib 基础：从折线图到第一条拟合曲线
2. 探索真实数据集：从一维直方图到多维散点图
3. 多子图排版与高清图片导出（`subplots` 与 `savefig`）
4. 用 Seaborn 快速绘制统计图（分组分布、箱线图、相关性热图与 `pairplot`）

In [ ]:
# colab-setup (本地运行时自动跳过)
import os, sys

if "google.colab" in sys.modules:
    _repo_dir = "/content/ai-for-materials"
    if not os.path.exists(_repo_dir):
        os.system("git clone --depth 1 https://github.com/Zeshengliu213/ai-for-materials.git " + _repo_dir)
    os.chdir(os.path.join(_repo_dir, "01 - Basics"))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---
## 1. Matplotlib 基础：从折线图到第一条拟合曲线

### 1.1 画一张基础折线图

我们先从一组最简单的数据开始。并苯（acene）系列——苯、萘、蒽、并四苯、并五苯——随着稠合苯环数 $N$ 从 1 增加到 5，$\pi$ 共轭体系逐渐拉长，光学能隙（Optical Gap）随之减小。我们把环数和能隙（近似值）放进两个 NumPy 数组里画出来：

In [ ]:
n_rings = np.array([1, 2, 3, 4, 5])
gap = np.array([4.9, 4.0, 3.3, 2.6, 2.1])    # 第一单重激发态能量 (eV)，示意约数

plt.plot(n_rings, gap, "o-")                 # "o-" 表示画圆点并用实线连起来
plt.xlabel("Number of fused rings (N)")
plt.ylabel("Optical gap (eV)")
plt.title("Acenes: optical gap vs. conjugation length")
plt.show()

用 Matplotlib 画图的基本套路就是上面这三步：
1. 用 `plt.plot(x, y, 格式)` 把数据画上画布（比如 `"o-"` 是圆点实线，`"s--"` 是方块虚线，`"o"` 是只画点不连线）；
2. 用 `plt.xlabel()`、`plt.ylabel()` 标明坐标轴名称和**物理量单位**（养成好习惯：没有坐标轴标签和单位的图是不合格的）；
3. 最后调用 `plt.show()` 把整张图渲染出来。

### 1.2 在同一张图上叠加经验拟合曲线

量子化学里最简单的“一维箱中粒子（自由电子气）”模型告诉我们，共轭多烯或并苯的能隙大致与共轭长度的倒数 $1/N$ 成正比：
$$E_{\text{gap}}(N) \approx a + \frac{b}{N}$$

我们可以用 NumPy 的多项式拟合函数 `np.polyfit(x, y, deg=1)`，直接对 $1/N$ 和 `gap` 做一次线性拟合（这其实就是你在本课程里建立的**第一个回归模型**！），然后把拟合出来的平滑曲线和实验数据点画在同一张图上：

In [ ]:
# 对 x = 1/n_rings, y = gap 做一阶线性拟合，返回 [斜率 b, 截距 a]
b, a = np.polyfit(1 / n_rings, gap, deg=1)
print(f"拟合公式: gap ≈ {a:.2f} + {b:.2f} / N")

# 生成从 N = 1 到 N = 10 的 100 个密集点，用来画平滑曲线
n_fine = np.linspace(1, 10, 100)

plt.plot(n_rings, gap, "o", label="Experimental (approx.)")
plt.plot(n_fine, a + b / n_fine, "-", label=f"Fit: {a:.2f} + {b:.2f}/N")
plt.axhline(a, color="gray", ls=":", label=f"N -> inf limit ({a:.2f} eV)")

plt.xlabel("Number of fused rings (N)")
plt.ylabel("Optical gap (eV)")
plt.legend()    # 显示图例（自动读取上面每个 plot 里的 label）
plt.show()

> **值得思考的一个问题（关于“外推”）**：
> 在 $N = 1 \sim 5$ 的范围内，这条经验公式拟合得相当准。但如果顺着蓝线往右看，当 $N \to \infty$（无限长并苯）时，公式预测能隙会收敛到约 $1.50\text{ eV}$。这个**外推（Extrapolation）**结论可靠吗？
>
> 实际上，当并苯超过 6 个环以后，电子关联效应和开壳层双自由基特征会变得非常显著，简单的 $1/N$ 模型会严重失效，而且长并苯在空气中极易氧化分解。**在已知数据范围之外盲目相信模型的外推预测，是材料机器学习中最常踩的坑之一**（我们在第 2 章讲适用域、第 4 章讲贝叶斯优化的不确定性时会深入探讨）。

---
## 2. 探索真实数据集：从一维分布到多维散点图

现在我们把上一节的 **AqSolDB 水溶解度数据集**（9482 个分子）读进来，看看怎么用图形探索一张大表。

In [ ]:
import os

DATA_FILE = "data/solubility.csv"
if not os.path.exists(DATA_FILE):
    DATA_FILE = "solubility.csv"
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        print("请上传 solubility.csv")
        files.upload()
    except ImportError:
        raise FileNotFoundError("找不到 solubility.csv，请确保它在 data/ 目录下")

df = pd.read_csv(DATA_FILE)
print(f"读取成功！共 {len(df)} 个分子。")

### 2.1 一维分布：直方图（`plt.hist`）

拿到一个目标性质（比如我们要预测的 `Solubility`），首先要看它在整个数据集里是怎么分布的。`plt.hist()` 会把数值区间切分成若干个小柱子（`bins`），统计落入每个区间的分子个数：

In [ ]:
plt.figure(figsize=(6, 4))                         # 设置画布尺寸：(宽, 高)，单位英寸
plt.hist(df["Solubility"], bins=50, edgecolor="white")
plt.xlabel("logS (log mol/L)")
plt.ylabel("Number of molecules")
plt.title("Distribution of aqueous solubility in AqSolDB")
plt.show()

从直方图可以一眼看出：绝大多数化合物的 $\log S$ 集中在 $-5$ 到 $0$ 之间，但左侧拖着一条一直延伸到 $-13$ 的长尾巴（正是我们在 `01b` 里排出来的那些大分子量疏水化合物）。

### 2.2 二维关系：散点图（`plt.scatter`）

看完了单个变量的分布，接下来看**两个变量之间的关系**。化学直觉告诉我们，油水不相溶：分子的疏水常数 `MolLogP` 越大（越亲油），在水里的溶解度 `Solubility` 应该越低。我们用散点图 `plt.scatter(x, y)` 把这 9482 个分子画出来验证一下：

In [ ]:
plt.figure(figsize=(6, 4.5))
plt.scatter(df["MolLogP"], df["Solubility"], s=5, alpha=0.3)
plt.xlabel("MolLogP (lipophilicity)")
plt.ylabel("logS (log mol/L)")
plt.title("Aqueous solubility vs. MolLogP")
plt.show()

注意我们在 `plt.scatter` 里传了两个非常实用的美化参数：
- `s=5`：把点的面积调小；
- `alpha=0.3`：把点的不透明度设为 30%（半透明）。

当数据点有好几千个时，如果不调小 `s` 和 `alpha`，中间密集的点就会完全糊成一团实心的死黑块，根本看不出数据真正扎堆的核心区域在哪里。

### 2.3 用颜色引入第三个维度

在二维散点图上，我们还可以把点的颜色 `c` 绑定到第三个变量（比如芳香环数 `NumAromaticRings`），再配上一根颜色条 `plt.colorbar()`，就能在一张平面图上同时观察三个物理量的联动关系：

In [ ]:
plt.figure(figsize=(6.8, 4.5))
# .clip(upper=6) 把大于 6 的环数统一截断到 6，避免极个别超大分子拉偏整个颜色条
sc = plt.scatter(
    df["MolLogP"],
    df["Solubility"],
    c=df["NumAromaticRings"].clip(upper=6),
    s=6,
    alpha=0.5,
    cmap="viridis",
)
plt.colorbar(sc, label="Number of aromatic rings (6 = 6+)")
plt.xlabel("MolLogP")
plt.ylabel("logS (log mol/L)")
plt.show()

可以看到，随着颜色从深紫（0 个芳香环）变成亮黄（多芳香环），数据点明显向**右下方**（高 `MolLogP`、低 `logS`）迁移。

---
## 3. 多子图排版与高清图片导出

写论文或做汇报时，我们经常需要把好几张小图并排拼成一张组图（Figure 1a, 1b, 1c）。这时候用 `fig, axes = plt.subplots(行数, 列数, figsize=(...))` 最规范：
- `fig` 代表整张大画布（用来调整全局间距、保存图片）；
- `axes` 是一个包含各个小图坐标轴对象的数组（`axes[0]` 是第一张小图，`axes[1]` 是第二张小图……）。

> **唯一需要注意的语法变化**：在单独的 `axes[i]` 上画图时，设坐标轴标签和范围的方法名要在前面加个 `set_`——比如 `plt.xlabel(...)` 变成了 `axes[0].set_xlabel(...)`，`plt.xlim(...)` 变成了 `axes[0].set_xlim(...)`。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))

# 子图 0：分子量分布直方图
axes[0].hist(df["MolWt"], bins=50, range=(0, 800), edgecolor="white")
axes[0].set_xlabel("Molecular weight (g/mol)")
axes[0].set_ylabel("Count")
axes[0].set_title("(a) MolWt distribution")

# 子图 1：溶解度 vs 分子量
axes[1].scatter(df["MolWt"], df["Solubility"], s=3, alpha=0.25)
axes[1].set_xlim(0, 800)
axes[1].set_xlabel("Molecular weight (g/mol)")
axes[1].set_ylabel("logS (log mol/L)")
axes[1].set_title("(b) logS vs. MolWt")

# 子图 2：溶解度 vs 极性表面积 TPSA
axes[2].scatter(df["TPSA"], df["Solubility"], s=3, alpha=0.25, color="tab:orange")
axes[2].set_xlim(0, 300)
axes[2].set_xlabel("TPSA (Å²)")
axes[2].set_ylabel("logS (log mol/L)")
axes[2].set_title("(c) logS vs. TPSA")

fig.tight_layout()       # 自动调整子图间距，防止坐标轴文字互相重叠
plt.show()

画好的图可以用 `fig.savefig("文件名", dpi=300)` 保存到硬盘上。投期刊通常要求位图分辨率至少 `dpi=300`；如果希望放进论文或 PPT 里无限放大都不模糊，可以直接把后缀改成 `.pdf` 或 `.svg` 导出为矢量图：

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(df["MolLogP"], df["Solubility"], s=4, alpha=0.3)
ax.set_xlabel("MolLogP")
ax.set_ylabel("logS (log mol/L)")
fig.tight_layout()

fig.savefig("solubility_vs_logp.png", dpi=300)
plt.close(fig)   # 存盘后关闭画布，不在页面上重复显示
print("已保存高清图片 solubility_vs_logp.png")

### 🏋️ 练习 01c-1

请使用 `plt.subplots(1, 2, figsize=(10, 4))` 画一张 1 行 2 列的组图：
- 左图（`axes[0]`）：极性表面积 `TPSA` 在 `0 ~ 300` 范围内的分布直方图（`bins=50`）；
- 右图（`axes[1]`）：可旋转键数 `NumRotatableBonds`（横轴，范围设为 `-1 ~ 40`）对水溶解度 `Solubility`（纵轴）的散点图。

记得给两张子图都加上合适的英文坐标轴标签，并用 `fig.tight_layout()` 调整布局。

In [ ]:
# 在这里写你的代码


---
## 4. 用 Seaborn 快速绘制统计图

如果我们需要按类别分组画图、画箱线图、或者画相关性矩阵，用纯 Matplotlib 得自己写循环去拆分数据、手动调颜色和图例。而 **`seaborn`** 专门为此设计：只要把整个 `DataFrame` 传给 `data=df`，再用列名指定 `x`、`y` 和分组颜色 `hue`，剩下的脏活累活它全包了。

### 4.1 分组直方图（`sns.histplot`）

我们先复用 `01b` 里的函数，给每个分子打上溶解度等级标签 `sol_class`，然后用一行 `sns.histplot` 看看不同溶解度等级的分子在 `MolLogP` 上是怎么分布的：

In [ ]:
def solubility_class(logs):
    if logs >= 0:
        return "highly soluble"
    elif logs >= -2:
        return "soluble"
    elif logs >= -4:
        return "slightly soluble"
    return "insoluble"

df["sol_class"] = df["Solubility"].apply(solubility_class)

plt.figure(figsize=(6.5, 4))
sns.histplot(
    data=df,
    x="MolLogP",
    hue="sol_class",
    bins=50,
    element="step",
    hue_order=["highly soluble", "soluble", "slightly soluble", "insoluble"],
)
plt.xlim(-10, 12)
plt.show()

### 4.2 箱线图（`sns.boxplot`）：不仅看均值，更要看组内涨落

在 `01b` 里，我们通过 `groupby("NumAromaticRings")` 发现芳香环从 0 增加到 5 时，平均 $\log S$ 逐步下降。但每组内部的分子溶解度到底有多分散？

用箱线图 `sns.boxplot` 可以把每一组的分布完整呈现出来：
- 箱子中间的横线是**中位数（50% 分位数）**；
- 箱子的上下边界分别是 **75% 和 25% 分位数**（箱子高度代表中间一半数据的跨度）；
- 上下伸出的须线之外的散点则是**离群值（Outliers）**。

In [ ]:
few_rings = df[df["NumAromaticRings"] <= 5]

plt.figure(figsize=(7, 4.2))
sns.boxplot(data=few_rings, x="NumAromaticRings", y="Solubility", color="lightsteelblue")
plt.xlabel("Number of aromatic rings")
plt.ylabel("logS (log mol/L)")
plt.show()

> **从箱线图看建模难度**：
> 虽然随着芳香环数增加，箱子整体一直在往下走（印证了“共轭环越多越难溶”的大趋势），但你看每一个箱子自身的上下跨度——哪怕同是含 2 个芳香环的分子，$\log S$ 也能从 $+1$ 跨到 $-10$！
>
> 这说明：**单凭“芳香环数”这一个特征，远远不足以把溶解度预测准**。因为环上接的是极性羟基/羧基，还是长链烷基/卤素，会让溶解度差出好几个数量级。这也正是后面我们需要综合十几个描述符（乃至用分子指纹和图神经网络）来建模的原因。

### 4.3 带线性回归趋势线的散点图（`sns.regplot`）

想在散点图上顺手叠加上一条最小二乘线性回归线，可以用 `sns.regplot`。为了避免近万个点显得太拥挤，我们先用 `df.sample(1000, random_state=0)` 随机抽样 1000 个分子（设定随机数种子 `random_state=0` 可以保证每次跑出来的抽样结果完全一致）：

In [ ]:
plt.figure(figsize=(6, 4.2))
sns.regplot(
    data=df.sample(1000, random_state=0),
    x="MolLogP",
    y="Solubility",
    scatter_kws={"s": 6, "alpha": 0.4},
    line_kws={"color": "crimson"},
)
plt.xlabel("MolLogP")
plt.ylabel("logS (log mol/L)")
plt.show()

### 4.4 相关性热图（`sns.heatmap`）

表里有十几个数值描述符，到底哪些描述符跟 `Solubility` 关系最密切？哪些描述符自己内部就高度绑定？

我们可以用 `df[cols].corr()` 一次性算出所有列两两之间的 **Pearson 线性相关系数 $r$**（取值在 $-1$ 到 $+1$ 之间：越接近 $+1$ 正相关越强，越接近 $-1$ 负相关越强，接近 $0$ 说明没有线性关系），然后交给 `sns.heatmap` 画成一目了然的红蓝热图：

In [ ]:
cols = [
    "Solubility", "MolWt", "MolLogP", "TPSA", "NumAromaticRings",
    "NumHDonors", "NumHAcceptors", "NumRotatableBonds", "RingCount"
]
corr = df[cols].corr()

plt.figure(figsize=(7.5, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1, square=True)
plt.title("Pearson correlation matrix")
plt.show()

### 🏋️ 练习 01c-2

1. 观察上面的相关性热图（第一行或第一列）：在这 8 个分子描述符中，与 `Solubility` **线性相关性最强**（$|r|$ 最大）的是哪一个？是正相关还是负相关？
2. 在下面的代码格里运行 `corr["Solubility"].drop("Solubility").abs().sort_values(ascending=False)`，按相关性绝对值从大到小排个序。
3. 看看热图里 `MolWt`（分子量）和 `RingCount`（总环数）之间的相关系数 $r$ 是多少？再看看 `TPSA`（极性表面积）和 `NumHAcceptors`（氢键受体数）之间的 $r$ 是多少？如果两个输入特征自己就高度相关（统计学上称为**多重共线性**），后面拿它们一起去做线性回归时可能会带来什么问题？

In [ ]:
# 在这里写你的代码


### 4.5 矩阵散点图（`sns.pairplot`）：一眼看尽两两关系

如果挑出了三四个核心变量，想同时看清它们各自的分布（对角线）以及任意两个变量之间的散点图（非对角线），`sns.pairplot` 一行代码就能生成全套矩阵图：

In [ ]:
sub = df.sample(600, random_state=0)
sns.pairplot(
    sub,
    vars=["Solubility", "MolLogP", "MolWt", "TPSA"],
    hue="sol_class",
    plot_kws={"s": 10, "alpha": 0.6},
    diag_kind="kde",
    hue_order=["highly soluble", "soluble", "slightly soluble", "insoluble"],
)
plt.show()

---
## 小结

这一节我们掌握了科研绘图的完整工具箱：
- **基础绘图（Matplotlib）**：用 `plt.plot` 画趋势与拟合曲线，用 `plt.hist` 看单变量分布，用 `plt.scatter`（配合透明度 `alpha` 与颜色映射 `cmap`）观察多变量关系，并用 `plt.subplots` 排版多子图、`savefig(dpi=300)` 导出论文级图片；
- **统计可视化（Seaborn）**：用 `histplot` 和 `boxplot` 对比不同分组的分布与涨落，用 `heatmap(df.corr())` 诊断特征与目标的相关性以及特征之间的多重共线性；
- **两个重要的建模伏笔**：
  1. 经验公式在训练范围之外的**外推风险**（长并苯能隙拟合）；
  2. 单一特征（如芳香环数）组内方差极大，以及多个特征之间存在冗余共线性（如 `TPSA` 与 `NumHAcceptors`）。

**下一节（`01d_rdkit_basics.ipynb`）**：到现在为止，我们一直在使用别人预先算好的 `MolWt`、`MolLogP`、`TPSA` 等表格列。如果手头只有分子的化学结构（SMILES），怎样自己动手把一个分子变成计算机能读懂的数字描述符和分子指纹？这就是下一节 **RDKit** 要解决的核心问题。